# Module C, sequence 3: analysis (corrected version)

Corrected version: the notebook to complete is on the branch `main`. The steps follow the TP 3 of the handout of the module C.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.h5modules.data_saving import DataLoader

DATA = Path('../data')  # a pathlib.Path: the operator / joins path parts (DATA / 'file.h5')
SCAN = '/RawData/Scan000/Detector000/Data0D/CH00/Data00'  # one scan
SCAN8 = '/RawData/Scan001/Detector000/Data0D/CH00/Data00'  # the same scan repeated 8 times (scan_average = 8)


def model(x, amp, x0, fwhm, offset):
    return amp * gauss1D(x, x0, fwhm) + offset  # FWHM is the width parameter of gauss1D

## 1. Load the scan file

In [ ]:
with DataLoader(DATA / 'scan_photodiode.h5') as loader:
    scan = loader.load_data(SCAN, load_all=True)
    scan8 = loader.load_data(SCAN8, load_all=True)
print(scan, scan.axes)
print(scan8, scan8.axes)

## 2. Look at it and find the lines

In [ ]:
mean8 = scan8.mean(0)
wavelength = mean8.get_axis_from_index(0)[0].get_data()
fig, ax = plt.subplots()
ax.plot(wavelength, scan.data[0], '.', label='one scan')
ax.plot(wavelength, mean8.data[0], '-', label='mean of 8 scans')
ax.set_xlabel('wavelength (nm)')
ax.legend()
peaks, _ = find_peaks(mean8.data[0], height=1.0)
print(wavelength[peaks])

## 3. Crop around the main line and fit with scipy (uncertainties from the covariance)

In [ ]:
window = mean8.vnav[505.:555.]
window.nav_indexes = ()
x = window.get_axis_from_index(0)[0].get_data()
y = window.data[0]
p0 = [y.max() - y.min(), x[np.argmax(y)], 5., y.min()]
popt, pcov = curve_fit(model, x, y, p0=p0)
perr = np.sqrt(np.diag(pcov))
for name, value, error in zip(['amplitude', 'center (nm)', 'FWHM (nm)', 'offset'], popt, perr):
    print(f'{name}: {value:.3f} +/- {error:.3f}')

## 4. The same fit with DataWithAxes.fit

In [ ]:
fit = window.fit(model, p0)
print(fit.fit_coeffs[0])
assert np.allclose(fit.fit_coeffs[0], popt, rtol=1e-3)

## 5. The axis in eV

In [ ]:
axis_ev = window.get_axis_from_index(0)[0].units_as('eV', inplace=False, context='spectroscopy')
window_ev = window.deepcopy()
window_ev.axes[0] = axis_ev
x_ev = axis_ev.get_data()
y_ev = window_ev.data[0]
p0_ev = [10., x_ev[np.argmax(y_ev)], 0.02, y_ev.min()]
popt_ev, pcov_ev = curve_fit(model, x_ev, y_ev, p0=p0_ev)
print(axis_ev.units, popt_ev[1], popt_ev[2], np.sqrt(np.diag(pcov_ev))[1:3])

## 6. One spectrum with Naverage = 200: the errors come with the data (PyMoDAQ 5.3.9 or later)

In [ ]:
with DataLoader(DATA / 'spectrum_naverage200.h5') as loader:
    spectrum = loader.load_all('/RawData/Detector000').get_data_from_name('Spectrum')
print(spectrum.errors[0][:3])

Plot the spectrum with its errors in the PyMoDAQ viewer (needs a display): where is the band the widest?

In [ ]:
%gui qt
import pymodaq_gui
spectrum.plot('qt')

## 7. Going further (open, for those who are done)

No hole to fill here and no single answer: pick one or several items, write your own cells, and keep what you find
in your handout. Everything you need is in the files of the `data` folder and in the objects above.

1. **The whole scan.** Fit `mean8` without cropping (three gaussians and one background, or one gaussian and the
   whole scan). Compare the background with the one of the cropped fit of the part 3: what do the satellite lines do?
   You will meet a point that the cropped window did not contain: look at `np.isnan(scan8.data[0])` and at what
   `curve_fit` says.
2. **Uncertainties from the repetitions.** `scan8.mean_with_errors(0)` puts the standard error of the mean of the 8
   repetitions in `errors`. Crop it again (the errors follow the slicing), plot it with `errorbar`, fit it with
   `curve_fit(..., sigma=window.errors[0], absolute_sigma=True)` and compute the reduced chi2. If it is larger than 1,
   are the error bars too small or is the model incomplete?
3. **Two ways to get a standard error.** Load `continuous_spectra.h5`
   (`'/RawData/Detector000/Data1D/CH00/EnlData00'`) and compare `series.mean_with_errors(0).errors[0]` with the
   `errors` of the spectrum of the part 6, at the top of the line and far from it. They are close at both places but
   much smaller far from the line: why? What does the standard deviation at the top, compared with the one far from
   the line, say about the origin of the noise?
4. **Model the drift.** In the continuous recording, fit the intensity at the peak versus the acquisition number with
   a sinusoid. What are its period (in acquisitions) and its relative depth? Divide the series by the fitted drift
   and compute again the standard deviation at the peak: what is left?
5. **The fit as a data object.** Build a `DataCalculated` holding the data of `window` and the fitted curve as two
   channels (same axis, labels 'data' and 'fit'), plot it with the PyMoDAQ viewer (`import pymodaq_gui` first, then
   `%gui qt` and `.plot('qt')`), then save it with its errors in an HDF5 file and open it in H5Browser.
6. **Navigation and signal on the series.** `series` has a navigation axis (time) and a signal axis (wavelength).
   Keep 525 to 532 nm on the signal axis with `vsig`: why does PyMoDAQ refuse (look at `series.distribution`)? Do it
   with `isig` instead (the axis has a `find_index` method), sum over the signal axis to get one number per
   acquisition, and compare with the intensity at the peak. Then keep the first 50 acquisitions only.
7. **Your own measurement.** Record a scan of the photodiode yourself in the DAQ_Scan (module A), with the satellite
   lines and the drift switched on in the settings of the plugin, and run this notebook on your file.

In [ ]:
from pymodaq_data.data import DataCalculated

# 1. the whole scan: the finite points only (one point of the repetition 7 is NaN), three lines and one background
full = mean8.deepcopy()
full.nav_indexes = ()
x_full = full.get_axis_from_index(0)[0].get_data()
y_full = full.data[0]
finite = np.isfinite(y_full)


def three_lines(x, a1, x1, w1, a2, x2, w2, a3, x3, w3, offset):
    return a1 * gauss1D(x, x1, w1) + a2 * gauss1D(x, x2, w2) + a3 * gauss1D(x, x3, w3) + offset


p0_three = [1.5, 471., 4., 10., 529., 4., 1., 590., 4., 0.3]
p_three, pcov_three = curve_fit(three_lines, x_full[finite], y_full[finite], p0=p0_three)
popt_full, pcov_full = curve_fit(model, x_full[finite], y_full[finite], p0=p0)
print('background, cropped:', popt[3], ' whole scan, one gaussian:', popt_full[3], ' three lines:', p_three[9])

# 2. the uncertainties from the repetitions, the weighted fit and the reduced chi2
mean8_err = scan8.mean_with_errors(0)
window_err = mean8_err.vnav[505.:555.]
window_err.nav_indexes = ()
popt_w, pcov_w = curve_fit(model, x, window_err.data[0], p0=p0, sigma=window_err.errors[0], absolute_sigma=True)
chi2_red = np.sum(((window_err.data[0] - model(x, *popt_w)) / window_err.errors[0]) ** 2) / (len(x) - len(popt_w))
print(popt_w, np.sqrt(np.diag(pcov_w)))
print('reduced chi2:', chi2_red)  # larger than 1: the intensity drifts during the scan, which the errors do not contain
fig, ax = plt.subplots()
ax.errorbar(x, window_err.data[0], yerr=window_err.errors[0], fmt='.', label='mean of 8 scans')
ax.plot(x, model(x, *popt_w), '-', label='weighted fit')
ax.set_xlabel('wavelength (nm)')
ax.legend()

# 3. the standard error of the continuous recording against the one of the DAQ_Viewer
with DataLoader(DATA / 'continuous_spectra.h5') as loader:
    series = loader.load_data('/RawData/Detector000/Data1D/CH00/EnlData00', load_all=True)
n_spectra = series.shape[0]
sem_series = series.mean_with_errors(0).errors[0]
i_peak = int(np.argmax(spectrum.data[0]))
print('standard error at the peak, Naverage=200:', spectrum.errors[0][i_peak], ' series:', sem_series[i_peak])
print('standard error far from the peak, Naverage=200:', spectrum.errors[0][:10].mean(), ' series:', sem_series[:10].mean())
# far from the line only the additive noise of the detector is left: sem = std / sqrt(200), about 0.01;
# at the top the std is ten times larger because the intensity drift is proportional to the signal

# 4. the drift: a sinusoid of the intensity at the peak versus the acquisition number
def drift(n, mean, depth, period, phase):
    return mean * (1 + depth * np.sin(2 * np.pi * n / period + phase))


peak = series.data[0][:, i_peak]
n = np.arange(n_spectra)
p_drift, pcov_drift = curve_fit(drift, n, peak, p0=[peak.mean(), 0.2, 35., 0.])
print('period (acquisitions):', abs(p_drift[2]), ' depth:', abs(p_drift[1]))
corrected = peak / (drift(n, *p_drift) / p_drift[0])
print('std at the peak, raw:', peak.std(ddof=1), ' drift removed:', corrected.std(ddof=1))

# 5. the fit as a data object, saved with its errors
fit_dwa = DataCalculated('line_fit', data=[window_err.data[0], model(x, *popt_w)], axes=window_err.axes,
                         labels=['data', 'fit'], errors=[window_err.errors[0], np.zeros_like(x)])
print(fit_dwa)

# 6. a continuous recording is a spread data: slice the signal axis by indexes
print(series.distribution)
wavelength_axis = series.get_axis_from_index(1)[0]
band = series.isig[wavelength_axis.find_index(525.):wavelength_axis.find_index(532.)]
band_sum = band.sum(1)  # one number per acquisition
print(band, band_sum, series.inav[:50])